# Preprocess

DEEP-PSMA zips → `data/processed/v1/<case>/` (PET in SUV, CT on the PET grid, TTB, TotalSegmentator labels, body mask, `qc.json`), then the frozen split `split_v1.json`. Resumes per patient: re-run after a disconnect. CPU runtime is enough.

Afterwards, the split lives on Drive at `data/processed/v1/split_v1.json`; it gets committed to `splits/` from there.

In [ ]:
# 1 — bootstrap: fresh code from GitHub (re-run after every push)
!git clone -q https://github.com/sezaiemrekonuk/ct-guided-pet-interp.git /content/petinterp 2>/dev/null || git -C /content/petinterp pull -q
%pip install -q SimpleITK
import sys; sys.path.insert(0, '/content/petinterp/src')
%cd /content/petinterp

In [ ]:
# 2 — start the run
import petinterp.runtime as rt
run = rt.start('configs/p1_preprocess_deep_psma.yaml')

In [ ]:
# 3 — preprocess every case, then split
from petinterp.data.preprocess import preprocess, make_split
qcs = preprocess(run)
split = make_split(run, qcs)

In [ ]:
# 4 — QC overview: liver SUVmean across the cohort (the open question from 2026-08-26)
import pandas as pd
df = pd.DataFrame([{**q, **(q.get('organ_suvmean') or {})} for q in qcs])
display(df[['case','included','n_slices','ttb_ml','suv_max','liver','kidney_left','urinary_bladder','flag_liver_out_of_range','flag_psma_pattern']])
display(df['liver'].describe())